In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Any
from sklearn.metrics import precision_score#, recall_score, f1_score, roc_auc_score

from src.counterfactual_fraud_model import (
    OffPolicyEvaluationPipeline,
    OffPolicyEvaluationConfig,
    DataGeneratorConfig,
    LoggingPolicyConfig,
    CounterfactualEstimatorConfig,
    PipelineConfig
)

In [ ]:
# Create configuration for the pipeline
config = OffPolicyEvaluationConfig(
    data_generator=DataGeneratorConfig(
        # HACK: erase later
        sample_size=10_000,
        random_state=223
    ),
    logging_policy=LoggingPolicyConfig(
        cutoff=0.1,
        exploration_rate=0.05, 
        random_state=223
    ),
    counterfactual_estimator=CounterfactualEstimatorConfig(random_state=223),
    pipeline=PipelineConfig(include_data=True)
)

# Initialize pipeline with configuration
pipeline = OffPolicyEvaluationPipeline(config)


In [36]:
results = pipeline.run_pipeline(cutoff=0.3, exploration_rate=0.05, include_data=True)

In [37]:
investigate = results['data']
investigate.head()

,model_scores,is_fraud,propensity_score,model_action,policy_action
0,0.056512,0,1.0,allow,allow
1,0.084480,0,1.0,allow,allow
2,0.662165,0,0.0,block,block
3,0.012744,0,1.0,allow,allow
4,0.000006,0,1.0,allow,allow


In [38]:
investigate.groupby(['model_action', 'policy_action']).size()

model_action  policy_action
allow         allow            9470
block         allow              21
              block             509
dtype: int64

In [39]:
y_true = investigate['is_fraud']
y_pred = (investigate['model_action'] == 'block').astype(int)

precision_score(y_true, y_pred)

0.18867924528301888

In [43]:
filtered_data = investigate[investigate['policy_action'] == 'allow']

y_true = filtered_data['is_fraud']
y_pred = (filtered_data['model_action'] == 'block').astype(int)
weights = 1 / filtered_data['propensity_score']

precision_score(y_true, y_pred)

0.2857142857142857

In [44]:
filtered_data[filtered_data['propensity_score'] != 1]

,model_scores,is_fraud,propensity_score,model_action,policy_action
46,0.360226,0,0.05,block,allow
1435,0.341580,0,0.05,block,allow
1611,0.385575,0,0.05,block,allow
2017,0.731744,0,0.05,block,allow
2536,0.469460,0,0.05,block,allow
3681,0.744622,0,0.05,block,allow
4011,0.447179,1,0.05,block,allow
4088,0.684527,0,0.05,block,allow
4312,0.642192,1,0.05,block,allow
5041,0.307551,0,0.05,block,allow


In [46]:
results['ope_metrics']

{'precision': {'estimate': 0.045037109633592275,
  'std_error': 0.0021068730512994456,
  'ci_lower': 0.04089590569861466,
  'ci_upper': 0.049182187829153866,
  'bootstrap_samples': [0.04242296327564373,
   0.04276211593284764,
   0.04583377336571972,
   0.042753087723002216,
   0.04563219604943488,
   0.04435526454747069,
   0.04265202702702703,
   0.04743291781111346,
   0.04308797127468582,
   0.04454295967912181,
   0.04400590966652596,
   0.04612137203166227,
   0.04433653541644674,
   0.04552175749894381,
   0.04922881893091063,
   0.043157117231191304,
   0.04634712837837838,
   0.04424965677473862,
   0.04517627190204771,
   0.04381797064723894,
   0.04739786762377283,
   0.04381334459459459,
   0.04149070945945946,
   0.04140699271152424,
   0.045497730391639395,
   0.04109878499735869,
   0.0426430230103441,
   0.04703519712503964,
   0.05114116652578191,
   0.04277566539923954,
   0.04151262279497201,
   0.0439328334565424,
   0.04051915163026274,
   0.044620253164556964,
   